# VN History Chunk Exporter for RAG-SFT Dataset

Notebook này dùng sau khi bạn đã build xong corpus và có file:

```text
vn_history_rag_chunks.csv
```

Mục đích:

1. Mount Google Drive.
2. Đọc CSV chunk đã xuất từ notebook corpus.
3. Search/lọc các chunk theo keyword.
4. Xuất thành các file `.md`, `.jsonl`, `.csv` nhỏ để upload/paste cho ChatGPT tạo dataset RAG-SFT.
5. Tạo template CSV để tự viết dataset nếu muốn.

Output nằm trong:

```text
/content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/
```

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports và cấu hình đường dẫn

In [2]:
from pathlib import Path
import json
import re
import math
from typing import List, Optional

import pandas as pd
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive/vn_history_model_backups')

# File CSV đầy đủ từ notebook build corpus. File này cần có cột text nguyên văn.
CHUNKS_CSV_PATH = DRIVE_ROOT / 'rag_corpus_vn_history' / 'processed' / 'vn_history_rag_chunks.csv'

# Fallback nếu muốn đọc JSONL thay vì CSV.
CHUNKS_JSONL_PATH = DRIVE_ROOT / 'rag_corpus_vn_history' / 'processed' / 'vn_history_rag_chunks.jsonl'

OUTPUT_DIR = DRIVE_ROOT / 'rag_dataset_prep'
BATCH_DIR = OUTPUT_DIR / 'chunk_batches'
TEMPLATE_DIR = OUTPUT_DIR / 'dataset_templates'

for p in [OUTPUT_DIR, BATCH_DIR, TEMPLATE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('CHUNKS_CSV_PATH:', CHUNKS_CSV_PATH)
print('CHUNKS_JSONL_PATH:', CHUNKS_JSONL_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

CHUNKS_CSV_PATH: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.csv
CHUNKS_JSONL_PATH: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl
OUTPUT_DIR: /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep


## 2. Đọc file chunk CSV/JSONL

In [3]:
def read_chunks_table(csv_path: Path, jsonl_path: Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    jsonl_path = Path(jsonl_path)

    if csv_path.exists():
        print('Đang đọc CSV:', csv_path)
        df = pd.read_csv(csv_path)
    elif jsonl_path.exists():
        print('Không thấy CSV, đang đọc JSONL:', jsonl_path)
        df = pd.read_json(jsonl_path, lines=True)
    else:
        raise FileNotFoundError(f'Không tìm thấy CSV hoặc JSONL:\nCSV: {csv_path}\nJSONL: {jsonl_path}')

    if 'text' not in df.columns and 'text_preview' in df.columns:
        print('CẢNH BÁO: Không có cột text, chỉ có text_preview. Nên dùng file CSV đầy đủ.')
        df['text'] = df['text_preview']

    required = ['chunk_id', 'title', 'text']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Thiếu cột bắt buộc: {missing}. Columns hiện có: {list(df.columns)}')

    for c in ['source', 'source_type', 'url', 'chunk_index', 'word_len', 'char_len', 'history_score']:
        if c not in df.columns:
            df[c] = ''

    df['chunk_id'] = df['chunk_id'].astype(str)
    df['title'] = df['title'].fillna('').astype(str)
    df['text'] = df['text'].fillna('').astype(str)
    df['url'] = df['url'].fillna('').astype(str)
    df['source'] = df['source'].fillna('').astype(str)

    for c in ['word_len', 'char_len', 'history_score']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

    return df

df = read_chunks_table(CHUNKS_CSV_PATH, CHUNKS_JSONL_PATH)

print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(5))

Đang đọc CSV: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.csv
Shape: (124065, 14)
Columns: ['chunk_id', 'source', 'source_type', 'title', 'section', 'url', 'chunk_index', 'text', 'char_len', 'word_len', 'raw_record_index', 'hf_dataset', 'history_score', 'text_hash']


,chunk_id,source,source_type,title,section,url,chunk_index,text,char_len,word_len,raw_record_index,hf_dataset,history_score,text_hash
0,hf_wikipedia_ohio_0000_c99496304b2d,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,NaN,https://vi.wikipedia.org/wiki/Ohio,0,"Ohio (viết tắt là OH, viết tắt cũ là O.) là mộ...",3142,650,3,DataStudio/Viet-wikipedia,20,cf1828cee8479709
1,hf_wikipedia_ohio_0001_6cbf9e0d55dd,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,NaN,https://vi.wikipedia.org/wiki/Ohio,1,lý Sông Ohio là biên giới phía nam của Ohio (c...,3112,650,3,DataStudio/Viet-wikipedia,20,84144bd943d6baab
2,hf_wikipedia_ohio_0002_fca9b4d7e318,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,NaN,https://vi.wikipedia.org/wiki/Ohio,2,"Thái Bình Dương. Trong số đó: 0,8% (88.627 ngư...",1025,214,3,DataStudio/Viet-wikipedia,20,2a89f8fd4f17e857
3,hf_wikipedia_california_0000_b2cb2058f797,DataStudio/Viet-wikipedia,hf_wikipedia,California,NaN,https://vi.wikipedia.org/wiki/California,0,California (còn được người Việt gọi vắn tắt là...,3001,650,4,DataStudio/Viet-wikipedia,13,da0fd057ef75257e
4,hf_wikipedia_california_0001_0dc9baa3bc7e,DataStudio/Viet-wikipedia,hf_wikipedia,California,NaN,https://vi.wikipedia.org/wiki/California,1,trình độ trung học – tỉ lệ thấp nhất trên tổng...,3105,650,4,DataStudio/Viet-wikipedia,13,dd954e454a0e086e


## 3. Xem thống kê nhanh

In [4]:
print('Số chunk:', len(df))
print('Số title:', df['title'].nunique())

if 'source_type' in df.columns:
    print('\nSource type counts:')
    display(df['source_type'].value_counts().head(20))

if 'word_len' in df.columns:
    print('\nWord length stats:')
    display(df['word_len'].describe())

if 'history_score' in df.columns:
    print('\nHistory score stats:')
    display(df['history_score'].describe())

print('\nTop titles:')
display(df['title'].value_counts().head(30))

Số chunk: 124065
Số title: 33088

Source type counts:


,count
source_type,
hf_wikipedia,124065



Word length stats:


,word_len
count,124065.000000
mean,574.971249
std,146.497116
min,120.000000
25%,608.000000
50%,650.000000
75%,650.000000
max,650.000000



History score stats:


,history_score
count,124065.000000
mean,22.696804
std,21.354985
min,7.000000
25%,10.000000
50%,15.000000
75%,26.000000
max,273.000000



Top titles:


,count
title,
Chiến tranh Đông Dương,113
Nhạc Phi,99
Kinh tế Bắc Triều Tiên,86
Lịch sử Áo,78
Phaolô Nguyễn Văn Bình,78
Chiến tranh Việt Nam,78
Chiến tranh Xô–Đức,78
Lịch sử Trung Quốc,76
Liên Xô,76


## 4. Hàm search, xem chunk, xuất pack

In [5]:
def normalize_text(s: str) -> str:
    s = '' if s is None else str(s)
    s = s.lower()
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def search_chunks(
    df: pd.DataFrame,
    keywords: List[str],
    search_in_title: bool = True,
    search_in_text: bool = True,
    min_word_len: int = 80,
    max_word_len: Optional[int] = None,
    min_history_score: Optional[float] = None,
    top_n: int = 50,
) -> pd.DataFrame:
    work = df.copy()
    mask = pd.Series(False, index=work.index)

    for kw in keywords:
        kw_l = normalize_text(kw)
        if not kw_l:
            continue
        if search_in_title:
            mask = mask | work['title'].astype(str).str.lower().str.contains(re.escape(kw_l), na=False)
        if search_in_text:
            mask = mask | work['text'].astype(str).str.lower().str.contains(re.escape(kw_l), na=False)

    work = work.loc[mask].copy()

    if min_word_len is not None and 'word_len' in work.columns:
        work = work[work['word_len'] >= min_word_len]
    if max_word_len is not None and 'word_len' in work.columns:
        work = work[work['word_len'] <= max_word_len]
    if min_history_score is not None and 'history_score' in work.columns:
        work = work[work['history_score'] >= min_history_score]

    if len(work):
        joined_kw = '|'.join(re.escape(normalize_text(k)) for k in keywords if normalize_text(k))
        work['title_match'] = work['title'].astype(str).str.lower().str.contains(joined_kw, na=False).astype(int)
        if 'word_len' in work.columns:
            work['text_len_abs_from_350'] = (work['word_len'] - 350).abs()
        else:
            work['text_len_abs_from_350'] = 0

        sort_cols = ['title_match']
        ascending = [False]
        if 'history_score' in work.columns:
            sort_cols.append('history_score')
            ascending.append(False)
        sort_cols.append('text_len_abs_from_350')
        ascending.append(True)
        work = work.sort_values(sort_cols, ascending=ascending)

    return work.head(top_n).reset_index(drop=True)

def show_chunks(df_sel: pd.DataFrame, n: int = 10, text_chars: int = 800):
    if df_sel is None or len(df_sel) == 0:
        print('Không có chunk nào.')
        return
    view = df_sel.head(n).copy()
    view['text_preview'] = view['text'].astype(str).str.slice(0, text_chars)
    cols = ['chunk_id', 'title', 'source_type', 'source', 'url', 'chunk_index', 'word_len', 'history_score', 'text_preview']
    cols = [c for c in cols if c in view.columns]
    display(view[cols])

def show_full_chunk(row):
    if isinstance(row, pd.DataFrame):
        if len(row) == 0:
            print('Không tìm thấy chunk.')
            return
        row = row.iloc[0]
    print('=' * 120)
    print('chunk_id:', row.get('chunk_id', ''))
    print('title:', row.get('title', ''))
    print('source:', row.get('source', ''))
    print('url:', row.get('url', ''))
    print('chunk_index:', row.get('chunk_index', ''))
    print('word_len:', row.get('word_len', ''))
    print('history_score:', row.get('history_score', ''))
    print('=' * 120)
    print(row.get('text', ''))

def truncate_text(text: str, max_chars: int = 2200) -> str:
    text = '' if text is None else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + ' ... [ĐÃ CẮT BỚT]'

def make_chatgpt_markdown_pack(
    df_pack: pd.DataFrame,
    pack_title: str = 'VN History RAG Chunk Pack',
    max_chars_per_chunk: int = 2200,
) -> str:
    lines = []
    lines.append(f'# {pack_title}')
    lines.append('')
    lines.append('## Yêu cầu cho ChatGPT')
    lines.append('Dựa trên các chunk dưới đây, hãy tạo dataset RAG-SFT. Mỗi sample cần có: type, question, context, gold_answer, gold_evidence. Tạo cả grounded_qa, noisy_context, insufficient_context, false_premise nếu phù hợp. Không bịa ngoài nội dung chunk.')
    lines.append('')
    lines.append('## Format dataset mong muốn')
    lines.append('```json')
    lines.append('{')
    lines.append('  "id": "sample_0001",')
    lines.append('  "type": "grounded_qa | noisy_context | insufficient_context | false_premise",')
    lines.append('  "question": "...",')
    lines.append('  "context": [{"chunk_id": "...", "title": "...", "text": "..."}],')
    lines.append('  "gold_answer": "...",')
    lines.append('  "gold_evidence": ["chunk_id"]')
    lines.append('}')
    lines.append('```')
    lines.append('')
    lines.append('## Chunks')
    lines.append('')

    for i, (_, r) in enumerate(df_pack.iterrows(), start=1):
        lines.append(f'### CHUNK {i:03d}')
        lines.append(f'- chunk_id: `{r.get("chunk_id", "")}`')
        lines.append(f'- title: {r.get("title", "")}')
        lines.append(f'- source_type: {r.get("source_type", "")}')
        lines.append(f'- source: {r.get("source", "")}')
        lines.append(f'- url: {r.get("url", "")}')
        lines.append(f'- chunk_index: {r.get("chunk_index", "")}')
        lines.append(f'- word_len: {r.get("word_len", "")}')
        lines.append(f'- history_score: {r.get("history_score", "")}')
        lines.append('')
        lines.append('```text')
        lines.append(truncate_text(r.get('text', ''), max_chars=max_chars_per_chunk))
        lines.append('```')
        lines.append('')
    return '\n'.join(lines)

def export_selected_chunks(
    df_pack: pd.DataFrame,
    out_dir: Path,
    base_name: str = 'chunk_pack',
    max_chars_per_chunk: int = 2200,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    md_path = out_dir / f'{base_name}.md'
    jsonl_path = out_dir / f'{base_name}.jsonl'
    csv_path = out_dir / f'{base_name}.csv'

    md = make_chatgpt_markdown_pack(df_pack, pack_title=base_name, max_chars_per_chunk=max_chars_per_chunk)
    md_path.write_text(md, encoding='utf-8')

    keep_cols = ['chunk_id', 'title', 'source_type', 'source', 'url', 'chunk_index', 'word_len', 'char_len', 'history_score', 'text']
    keep_cols = [c for c in keep_cols if c in df_pack.columns]

    df_pack[keep_cols].to_csv(csv_path, index=False, encoding='utf-8-sig')
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for _, r in df_pack[keep_cols].iterrows():
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + '\n')

    print('Đã xuất:')
    print('-', md_path)
    print('-', jsonl_path)
    print('-', csv_path)
    return md_path, jsonl_path, csv_path

def export_batches_from_df(
    df_input: pd.DataFrame,
    out_dir: Path,
    base_name: str = 'batch',
    chunks_per_batch: int = 10,
    max_chars_per_chunk: int = 2200,
):
    paths = []
    total = len(df_input)
    if total == 0:
        print('Không có chunk để tách batch.')
        return paths
    num_batches = math.ceil(total / chunks_per_batch)
    for b in range(num_batches):
        part = df_input.iloc[b * chunks_per_batch : min((b + 1) * chunks_per_batch, total)].copy()
        name = f'{base_name}_{b+1:03d}'
        md_path, _, _ = export_selected_chunks(part, out_dir=out_dir, base_name=name, max_chars_per_chunk=max_chars_per_chunk)
        paths.append(md_path)
    return paths

## 5. Search thử theo keyword

In [6]:
# Sửa keyword ở đây theo chủ đề bạn muốn tạo dataset.
KEYWORDS = ['Bạch Đằng', 'Ngô Quyền', 'Nam Hán']

df_sel = search_chunks(
    df,
    keywords=KEYWORDS,
    min_word_len=80,
    min_history_score=None,
    top_n=30,
)

print('Keywords:', KEYWORDS)
print('Found:', len(df_sel))
show_chunks(df_sel, n=20, text_chars=900)

Keywords: ['Bạch Đằng', 'Ngô Quyền', 'Nam Hán']
Found: 30


,chunk_id,title,source_type,source,url,chunk_index,word_len,history_score,text_preview
0,hf_wikipedia_ngô_quyền_0008_117505677cc6,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,8,414,101,Tử làm thánh thành hoàng. Nhiều đường phố mang...
1,hf_wikipedia_ngô_quyền_0000_af223d790816,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,0,650,101,Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 n...
2,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,1,650,101,Tiết độ sứ cuối cùng trong thời kì Tự chủ. Như...
3,hf_wikipedia_ngô_quyền_0002_91f9c47eaa2a,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,2,650,101,thì thế được thua chưa biết ra sao. Nếu sai ng...
4,hf_wikipedia_ngô_quyền_0003_5da5ba8598ed,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,3,650,101,"Dương), Lê Lương ở Ái châu, Đinh Công Trứ (cha..."
5,hf_wikipedia_ngô_quyền_0004_8bba59d9751b,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,4,650,101,"miếu hiệu và thụy hiệu, sử sách xưa nay chỉ gọ..."
6,hf_wikipedia_ngô_quyền_0005_cadcb574e314,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,5,650,101,Đại Đường; do Ngô Thì Sĩ và Phan Huy Chú xác đ...
7,hf_wikipedia_ngô_quyền_0006_ea8128d42bb3,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,6,650,101,"là Dương hậu. Ông đã làm đảo chính, phế truất ..."
8,hf_wikipedia_ngô_quyền_0007_c6973c485128,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,7,650,101,cho việc đi lại và tiếp tế của nghĩa quân và n...
9,hf_wikipedia_trận_bạch_đằng_1288_0006_637c2118...,Trận Bạch Đằng (1288),hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Tr%E1%BA%ADn%20B...,6,517,84,trình dân sự từ thời văn hóa Đông Sơn? Trong t...


## 6. Xem full một chunk

In [7]:
# Đổi idx để xem chunk khác trong df_sel.
idx = 0

if len(df_sel):
    show_full_chunk(df_sel.iloc[idx])
else:
    print('df_sel đang rỗng.')

chunk_id: hf_wikipedia_ngô_quyền_0008_117505677cc6
title: Ngô Quyền
source: DataStudio/Viet-wikipedia
url: https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E1%BB%81n
chunk_index: 8
word_len: 414
history_score: 101
Tử làm thánh thành hoàng. Nhiều đường phố mang tên Ngô Quyền như tại quận Hoàn Kiếm và Hà Đông, Hà Nội, thành phố Thanh Hóa, thị xã Quảng Yên, thành phố Đà Nẵng, thành phố Quy Nhơn... Tên ông cũng là tên của một quận nội thành của Hải Phòng. Nhiều trường học ở Việt Nam cũng mang tên Ngô Quyền. Ảnh Xem thêm Khúc Thừa Dụ Dương Đình Nghệ Trận Bạch Đằng (938) Các bãi cọc trên sông Bạch Đằng Nhà Ngô Dương Tam Kha Dương Như Ngọc Ngô Xương Ngập Ngô Xương Văn Hậu Ngô Vương Chú thích Tham khảo Nhiều tác giả (1972), Đại Việt Sử ký Toàn thư, Cao Huy Giu phiên dịch, Nhà Xuất bản Khoa học Xã hội. Phan Bội Châu (1909), Việt Nam quốc sử khảo. Nhiều tác giả (1991), Lịch sử Việt Nam 1; Nhà Xuất bản Đại học và Giáo dục chuyên nghiệp. Tạ Chí Đại Trường (2009), Sơ thảo: Bài sử khác cho Việt Nam, ấn 

## 7. Chọn chunk và xuất pack hiện tại

In [8]:
# Lấy top N từ kết quả search hiện tại.
SELECT_TOP_N = 12
selected_df = df_sel.head(SELECT_TOP_N).copy()

# Đặt tên pack theo chủ đề.
PACK_NAME = 'pack_bach_dang_ngo_quyen'

print('Selected chunks:', len(selected_df))
show_chunks(selected_df, n=30, text_chars=500)

md_path, jsonl_path, csv_path = export_selected_chunks(
    selected_df,
    out_dir=BATCH_DIR,
    base_name=PACK_NAME,
    max_chars_per_chunk=2200,
)

print('\nBạn có thể mở file .md, copy nội dung hoặc upload file đó cho ChatGPT.')

Selected chunks: 12


,chunk_id,title,source_type,source,url,chunk_index,word_len,history_score,text_preview
0,hf_wikipedia_ngô_quyền_0008_117505677cc6,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,8,414,101,Tử làm thánh thành hoàng. Nhiều đường phố mang...
1,hf_wikipedia_ngô_quyền_0000_af223d790816,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,0,650,101,Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 n...
2,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,1,650,101,Tiết độ sứ cuối cùng trong thời kì Tự chủ. Như...
3,hf_wikipedia_ngô_quyền_0002_91f9c47eaa2a,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,2,650,101,thì thế được thua chưa biết ra sao. Nếu sai ng...
4,hf_wikipedia_ngô_quyền_0003_5da5ba8598ed,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,3,650,101,"Dương), Lê Lương ở Ái châu, Đinh Công Trứ (cha..."
5,hf_wikipedia_ngô_quyền_0004_8bba59d9751b,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,4,650,101,"miếu hiệu và thụy hiệu, sử sách xưa nay chỉ gọ..."
6,hf_wikipedia_ngô_quyền_0005_cadcb574e314,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,5,650,101,Đại Đường; do Ngô Thì Sĩ và Phan Huy Chú xác đ...
7,hf_wikipedia_ngô_quyền_0006_ea8128d42bb3,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,6,650,101,"là Dương hậu. Ông đã làm đảo chính, phế truất ..."
8,hf_wikipedia_ngô_quyền_0007_c6973c485128,Ngô Quyền,hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E...,7,650,101,cho việc đi lại và tiếp tế của nghĩa quân và n...
9,hf_wikipedia_trận_bạch_đằng_1288_0006_637c2118...,Trận Bạch Đằng (1288),hf_wikipedia,DataStudio/Viet-wikipedia,https://vi.wikipedia.org/wiki/Tr%E1%BA%ADn%20B...,6,517,84,trình dân sự từ thời văn hóa Đông Sơn? Trong t...


Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.csv

Bạn có thể mở file .md, copy nội dung hoặc upload file đó cho ChatGPT.


## 8. Tạo nhiều pack tự động theo chủ đề

In [9]:
TOPIC_KEYWORDS = {
    'bach_dang_ngo_quyen': ['Bạch Đằng', 'Ngô Quyền', 'Nam Hán'],
    'nha_dinh_tien_le': ['Đinh Bộ Lĩnh', 'Đại Cồ Việt', 'Lê Hoàn', 'Tiền Lê'],
    'nha_ly': ['nhà Lý', 'Lý Công Uẩn', 'Lý Thường Kiệt', 'sông Như Nguyệt'],
    'nha_tran_mong_nguyen': ['nhà Trần', 'Trần Hưng Đạo', 'Mông Nguyên', 'Bạch Đằng 1288'],
    'lam_son_le_loi': ['Lam Sơn', 'Lê Lợi', 'Nguyễn Trãi', 'Bình Ngô đại cáo'],
    'tay_son_quang_trung': ['Tây Sơn', 'Quang Trung', 'Ngọc Hồi', 'Đống Đa'],
    'can_vuong_phap_thuoc': ['Cần Vương', 'Hàm Nghi', 'Phan Đình Phùng', 'thực dân Pháp'],
    'cach_mang_thang_tam': ['Cách mạng tháng Tám', 'Việt Minh', 'Tuyên ngôn độc lập'],
    'dien_bien_phu': ['Điện Biên Phủ', 'Võ Nguyên Giáp', 'Geneva 1954'],
    'chong_my_1975': ['Mậu Thân 1968', 'Hiệp định Paris', 'Chiến dịch Hồ Chí Minh', '30 tháng 4'],
}

AUTO_TOP_N_PER_TOPIC = 12
created = []

for topic, kws in TOPIC_KEYWORDS.items():
    temp = search_chunks(df, keywords=kws, min_word_len=80, min_history_score=None, top_n=AUTO_TOP_N_PER_TOPIC)
    if len(temp) == 0:
        print('Không có chunk cho topic:', topic)
        continue
    paths = export_selected_chunks(temp, out_dir=BATCH_DIR, base_name=f'pack_{topic}', max_chars_per_chunk=2200)
    created.append((topic, len(temp), paths[0]))

print('\nCreated packs:')
for topic, n, path in created:
    print(f'- {topic}: {n} chunks -> {path}')

Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.csv
Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_dinh_tien_le.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_dinh_tien_le.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_dinh_tien_le.csv
Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_ly.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_ly.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_nha_ly.csv
Đã xuất:
- /content/drive/My

## 9. Tạo batch lớn từ các chunk score cao

In [10]:
# Tạo 100 chunk score cao, tách thành nhiều file nhỏ, mỗi file 10 chunk.
if 'history_score' in df.columns:
    big_df = df.sort_values(['history_score', 'word_len'], ascending=[False, False]).head(100).copy()
else:
    big_df = df.head(100).copy()

batch_paths = export_batches_from_df(
    big_df,
    out_dir=BATCH_DIR / 'top_score_batches',
    base_name='top_score_pack',
    chunks_per_batch=10,
    max_chars_per_chunk=2000,
)

print('\nBatch markdown files:')
for p in batch_paths:
    print('-', p)

Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_001.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_001.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_001.csv
Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_002.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_002.jsonl
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_002.csv
Đã xuất:
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_003.md
- /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/top_score_batches/top_score_pack_003.

## 10. Tạo template CSV để tự viết dataset

In [11]:
def create_manual_dataset_template(df_pack: pd.DataFrame, output_path: Path, questions_per_chunk: int = 3):
    rows = []
    for _, r in df_pack.iterrows():
        for _ in range(questions_per_chunk):
            rows.append({
                'sample_id': f'sample_{len(rows)+1:05d}',
                'type': 'grounded_qa',
                'question': '',
                'context_chunk_ids': r['chunk_id'],
                'context_titles': r['title'],
                'context_text': r['text'],
                'gold_answer': '',
                'gold_evidence': r['chunk_id'],
                'note': '',
            })
    out = pd.DataFrame(rows)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_path, index=False, encoding='utf-8-sig')
    print('Đã lưu template:', output_path)
    return out

template_path = TEMPLATE_DIR / f'{PACK_NAME}_manual_dataset_template.csv'
template_df = create_manual_dataset_template(selected_df, output_path=template_path, questions_per_chunk=3)
display(template_df.head(10))

Đã lưu template: /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/dataset_templates/pack_bach_dang_ngo_quyen_manual_dataset_template.csv


,sample_id,type,question,context_chunk_ids,context_titles,context_text,gold_answer,gold_evidence,note
0,sample_00001,grounded_qa,,hf_wikipedia_ngô_quyền_0008_117505677cc6,Ngô Quyền,Tử làm thánh thành hoàng. Nhiều đường phố mang...,,hf_wikipedia_ngô_quyền_0008_117505677cc6,
1,sample_00002,grounded_qa,,hf_wikipedia_ngô_quyền_0008_117505677cc6,Ngô Quyền,Tử làm thánh thành hoàng. Nhiều đường phố mang...,,hf_wikipedia_ngô_quyền_0008_117505677cc6,
2,sample_00003,grounded_qa,,hf_wikipedia_ngô_quyền_0008_117505677cc6,Ngô Quyền,Tử làm thánh thành hoàng. Nhiều đường phố mang...,,hf_wikipedia_ngô_quyền_0008_117505677cc6,
3,sample_00004,grounded_qa,,hf_wikipedia_ngô_quyền_0000_af223d790816,Ngô Quyền,Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 n...,,hf_wikipedia_ngô_quyền_0000_af223d790816,
4,sample_00005,grounded_qa,,hf_wikipedia_ngô_quyền_0000_af223d790816,Ngô Quyền,Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 n...,,hf_wikipedia_ngô_quyền_0000_af223d790816,
5,sample_00006,grounded_qa,,hf_wikipedia_ngô_quyền_0000_af223d790816,Ngô Quyền,Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 n...,,hf_wikipedia_ngô_quyền_0000_af223d790816,
6,sample_00007,grounded_qa,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,Ngô Quyền,Tiết độ sứ cuối cùng trong thời kì Tự chủ. Như...,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,
7,sample_00008,grounded_qa,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,Ngô Quyền,Tiết độ sứ cuối cùng trong thời kì Tự chủ. Như...,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,
8,sample_00009,grounded_qa,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,Ngô Quyền,Tiết độ sứ cuối cùng trong thời kì Tự chủ. Như...,,hf_wikipedia_ngô_quyền_0001_c00ce6aaa0d9,
9,sample_00010,grounded_qa,,hf_wikipedia_ngô_quyền_0002_91f9c47eaa2a,Ngô Quyền,thì thế được thua chưa biết ra sao. Nếu sai ng...,,hf_wikipedia_ngô_quyền_0002_91f9c47eaa2a,


## 11. In preview nội dung pack markdown để copy nhanh

In [12]:
print('Markdown path:', md_path)
content = Path(md_path).read_text(encoding='utf-8')
print(content[:4000])
print('\n...[cắt preview]...')

Markdown path: /content/drive/MyDrive/vn_history_model_backups/rag_dataset_prep/chunk_batches/pack_bach_dang_ngo_quyen.md
# pack_bach_dang_ngo_quyen

## Yêu cầu cho ChatGPT
Dựa trên các chunk dưới đây, hãy tạo dataset RAG-SFT. Mỗi sample cần có: type, question, context, gold_answer, gold_evidence. Tạo cả grounded_qa, noisy_context, insufficient_context, false_premise nếu phù hợp. Không bịa ngoài nội dung chunk.

## Format dataset mong muốn
```json
{
  "id": "sample_0001",
  "type": "grounded_qa | noisy_context | insufficient_context | false_premise",
  "question": "...",
  "context": [{"chunk_id": "...", "title": "...", "text": "..."}],
  "gold_answer": "...",
  "gold_evidence": ["chunk_id"]
}
```

## Chunks

### CHUNK 001
- chunk_id: `hf_wikipedia_ngô_quyền_0008_117505677cc6`
- title: Ngô Quyền
- source_type: hf_wikipedia
- source: DataStudio/Viet-wikipedia
- url: https://vi.wikipedia.org/wiki/Ng%C3%B4%20Quy%E1%BB%81n
- chunk_index: 8
- word_len: 414
- history_score: 101

```text
Tử l

# Cách dùng thực tế

Sau khi chạy notebook:

1. Vào Drive:

```text
vn_history_model_backups/rag_dataset_prep/chunk_batches/
```

2. Chọn một file `.md`, ví dụ:

```text
pack_bach_dang_ngo_quyen.md
```

3. Upload file đó vào ChatGPT và nhắn:

```text
Dựa vào các chunk trong file này, tạo cho tôi 30 sample dataset RAG-SFT.
Mỗi sample có type, question, context, gold_answer, gold_evidence.
Ưu tiên có noisy_context và insufficient_context.
```

Không nên đưa quá nhiều chunk một lúc. Mỗi pack khoảng 8–15 chunk là vừa.